# Задание 5. Исследовательский анализ SMS-данных (EDA)

**Цель:** изучить структуру очищенного корпуса SMS (`data/processed/sms_clean.csv`), оценить баланс классов, длину сообщений, частотные N-граммы и долю специальных токенов (`<url>`, `<phone>`, `<money>`, `!`), а также сформулировать количественно подтверждённые гипотезы для построения признаков ML-модели.


In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df_clean = pd.read_csv(PROJECT_ROOT / "data/processed/sms_clean.csv")
with (PROJECT_ROOT / "data/processed/sms_cleaning_report.json").open(encoding="utf-8") as f:
    clean_report = json.load(f)
with (PROJECT_ROOT / "data/processed/sms_eda_summary.json").open(encoding="utf-8") as f:
    eda_summary = json.load(f)

print(f"Исходное число записей: {clean_report['initial_count']}")
print(f"Удалено пустых после очистки: {clean_report['empty_removed']}")
print(f"Удалено дубликатов по text_clean: {clean_report['duplicates_removed']} (ham: {clean_report['duplicates_by_class']['0']}, spam: {clean_report['duplicates_by_class']['1']})")
print(f"Итоговый размер очищенного корпуса: {len(df_clean)} строк")
print("Распределение классов:", clean_report["clean_class_counts"])


Исходное число записей: 5574
Удалено пустых после очистки: 2
Удалено дубликатов по text_clean: 472 (ham: 324, spam: 148)
Итоговый размер очищенного корпуса: 5100 строк
Распределение классов: {'ham': 4501, 'spam': 599, 'ham_pct': 88.25, 'spam_pct': 11.75, 'imbalance_ratio': 7.51}


## 1. Анализ распределения и дисбаланса классов

График сохранён в `docs/figures/01_sms_class_distribution.png`.
- **Безопасные сообщения (`ham`, класс 0):** `4 501` шт. (`88,25%`)
- **Мошеннические / спам (`spam`, класс 1):** `599` шт. (`11,75%`)
- **Коэффициент дисбаланса:** `7,51 : 1` в пользу класса `ham`.

**Как учитывается дисбаланс классов при обучении (Задания 6–8):**
1. Стратифицированное разбиение `70% / 15% / 15%` (`train_test_split(..., stratify=labels)`), сохраняющее долю `spam` на уровне `11,74%–11,76%` во всех трёх выборках (`train=3570`, `val=765`, `test=765`).
2. Использование взвешивания классов `class_weight='balanced'` в `LogisticRegression`, `LinearSVC` и `RandomForestClassifier`.
3. Оптимизация гиперпараметров и выбор порога по метрике **F2-score** (где полнота `Recall` класса `spam` весит вдвое больше точности `Precision`) и контроль `PR-AUC`.


In [2]:
print("Статистика длины сообщений в символах:", eda_summary["char_length"])
print("Статистика длины сообщений в словах:", eda_summary["word_length"])
print("Доля сообщений со спец-токенами (%):", json.dumps(eda_summary["token_shares_pct"], ensure_ascii=False, indent=2))
print("Средняя доля заглавных букв (%):", eda_summary["caps_ratio_pct"])


Статистика длины сообщений в символах: {'ham_mean': 71.1, 'ham_median': 53.0, 'ham_std': 56.59, 'spam_mean': 137.67, 'spam_median': 148.0, 'spam_std': 29.91}
Статистика длины сообщений в словах: {'ham_mean': 14.28, 'ham_median': 11.0, 'spam_mean': 23.76, 'spam_median': 25.0}
Доля сообщений со спец-токенами (%): {
  "url": {
    "ham": 0.27,
    "spam": 17.36
  },
  "phone": {
    "ham": 0.02,
    "spam": 54.42
  },
  "money": {
    "ham": 0.27,
    "spam": 33.56
  },
  "exclamation": {
    "ham": 11.78,
    "spam": 46.91
  },
  "num": {
    "ham": 15.73,
    "spam": 87.31
  }
}
Средняя доля заглавных букв (%): {'ham_mean': 7.63, 'spam_mean': 16.66}


In [3]:
print("Топ-10 слов (ham):", eda_summary["top_20_words_ham"][:10])
print("Топ-10 слов (spam):", eda_summary["top_20_words_spam"][:10])
print("Топ-10 биграмм (ham):", eda_summary["top_20_bigrams_ham"][:10])
print("Топ-10 биграмм (spam):", eda_summary["top_20_bigrams_spam"][:10])


Топ-10 слов (ham): [['your', 381], ['get', 301], ['up', 288], ['ok', 251], ['go', 248], ['ll', 231], ['got', 228], ['know', 226], ['like', 224], ['good', 217]]
Топ-10 слов (spam): [['call', 278], ['your', 221], ['free', 183], ['txt', 130], ['stop', 115], ['ur', 113], ['text', 105], ['mobile', 101], ['reply', 91], ['claim', 87]]
Топ-10 биграмм (ham): [['let know', 40], ['pick up', 37], ['take care', 36], ['good morning', 24], ['new year', 21], ['last night', 19], ['hi hi', 18], ['wat time', 18], ['wake up', 18], ['ok lor', 18]]
Топ-10 биграмм (spam): [['your mobile', 43], ['please call', 31], ['po box', 23], ['find out', 22], ['call landline', 18], ['send stop', 18], ['guaranteed call', 17], ['prize guaranteed', 16], ['urgent your', 16], ['txt stop', 16]]


## 2. Ключевые наблюдения и выводы по результатам EDA (с подтверждением числами)

1. **Существенный дисбаланс классов (`7,51 : 1`):** после удаления `472` дубликатов (`324` в `ham` и `148` в `spam`) и `2` пустых строк корпус насчитывает `5 100` сообщений, из которых `4 501` (`88,25%`) — безопасные (`ham`) и `599` (`11,75%`) — мошеннические/спам (`spam`). Использование `Accuracy` в одиночку недопустимо, так как константный классификатор дал бы `88,25%` точности при `Recall = 0%`.
2. **Мошеннические SMS почти вдвое длиннее безопасных по числу символов:** средняя длина `spam`-сообщений составляет **`137,67` символа** (медиана — **`148,0`**, стандартное отклонение — `29,91`), тогда как у безопасных `ham`-сообщений средняя длина равна **`71,10` символа** (медиана — **`53,0`**). Мошеннические сообщения плотно концентрируются возле лимита одного стандартного GSM SMS (140–160 символов).
3. **Различие длины в словах (`+66,4%` в пользу спама):** среднее число слов в `spam` составляет **`23,76` слова** (медиана — **`25,0`**) против **`14,28` слова** (медиана — **`11,0`**) в `ham`. Это обосновывает включение ручного признака `char_length`.
4. **Телефонные номера (`<phone>`) и суммы (`<money>`) — сильнейшие маркеры спама:** токен `<phone>` присутствует в **`54,42%`** мошеннических сообщений и лишь в **`0,02%`** безопасных (разница более чем в 2700 раз). Токен денежной суммы `<money>` встречается в **`33,56%`** спам-сообщений против **`0,27%`** в `ham` (в 124 раза чаще).
5. **Интернет-ссылки (`<url>`) в 64 раза чаще встречаются в мошеннических SMS:** токен `<url>` обнаружен в **`17,36%`** сообщений класса `spam` и только в **`0,27%`** сообщений класса `ham`, что подтверждает необходимость выделения признака `has_url` и передачи ссылок во второй контур анализа URL.
6. **Эмоциональное давление через восклицательные знаки и заглавные буквы (`CAPS`):** знак `!` встречается в **`46,91%`** сообщений класса `spam` против **`11,78%`** в `ham` (в 4,0 раза чаще), а средняя доля заглавных букв (`caps_ratio`) в `spam` составляет **`16,66%`** против **`7,63%`** в `ham` (в 2,18 раза выше).
7. **Чёткое разделение лексического ядра и биграмм:** в `spam` доминируют императивы и триггеры социальной инженерии (`call` — 278, `free` — 183, `txt` — 130, `stop` — 115, `claim` — 87, `prize` — 70, `cash` — 61, `urgent` — 51; биграммы `your mobile` — 43, `please call` — 31, `prize guaranteed` — 16, `urgent your` — 16), тогда как в `ham` преобладает бытовая диалоговая лексика (`ok` — 251, `go` — 248, `good` — 217, `come` — 215; биграммы `let know` — 40, `pick up` — 37, `take care` — 36, `good morning` — 24).
